In [6]:
import pulp as PLP

import numpy as np

import itertools
class symmetric_tsp_dfj:
    """ This implementation of the symmetric TSP DFJ problem follows from
    the slipes in uge 10, TSP-formuleringer (pdf), slides 7 - 10"""

    def __init__(self, n, cost_matrix = None, x_coords = None, y_coords = None):
        self.n = n
        self.cost_matrix = cost_matrix

        if x_coords is not None and y_coords is not None and cost_matrix is None:
            self.cost_matrix = self.cost_from_coordinates(x_coords, y_coords)

        # ILP problem
        self.model = PLP.LpProblem(name = "SymmetricTSP_DFJ", sense = PLP.LpMinimize)
        # Define arcs,
        self.arcs = [(i,j) for i in range(n) for j in range(n) if i < j]
        # Definte subsets
        self.S = self.define_subsets()
        #### Variable definition ####

    def cost_from_coordinates(self, x_cords, y_cords):
        # Computes L2 distance between points, this can serve as a cost.
        n = len(x_cords)
        cost_matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                if i == j:
                    # Set diagonal to very large number
                    very_large_number = 10 ** 6
                    cost_matrix[i][j] = very_large_number
                else:
                    diff_x = x_cords[i] - x_cords[j]
                    diff_y = y_cords[i] - y_cords[j]
                    dist = np.sqrt(diff_x ** 2 + diff_y ** 2)
                    cost_matrix[i][j] = dist
        return cost_matrix

    def define_subsets(self):
        S = []
        for cardinality in range(3, self.n - 3):
            # Finds all combinations of the cardinality
            subset = itertools.combinations(list(range(self.n)), cardinality)
            # Return the set
            subset = set(subset)
            # Add the subset to S
            S = S + list(subset)
        return S

    def solve_and_print(self, one_indexed = True, quiet = True):
        ### CONSTRAINTS ###
        # Binary variable indicating whether arc (i,j) is used in the solution
        self.x = PLP.LpVariable.dicts("x", self.arcs, lowBound=0, upBound=1, cat=PLP.LpBinary)

        #### Obejctive function ####
        self.model += PLP.lpSum(self.cost_matrix[i][j] * self.x[(i, j)] for i, j in self.arcs), "Objective"

        ### Valence contraint, makes sure that each point it connected to two other points
        for j in range(self.n):
            self.model += PLP.lpSum(self.x[(i, j)] for i in range(j)) + \
                          PLP.lpSum(self.x[(j, i)] for i in range(j + 1, self.n)) \
                          == 2, f"valence{j}"

        ### Subtour elimination, DFJ uses subsets and we restrict the solution such that
        # There does not exist a S subset of {0, 1, 2, ..., n - 1 } such there is a closed
        # loops in the solution in the vertices contained in S
        for k, s in enumerate(self.S):
            self.model += PLP.lpSum(self.x[(i, j)] for i in s for j in s if i < j) <= len(
                s) - 1, f"SubtourElimination{k}"

        ### SOLVE ###

        self.model.solve(PLP.PULP_CBC_CMD(msg = 0 if quiet else 1))
        print("Status:", PLP.LpStatus[self.model.status])
        print("Objective value:", PLP.value(self.model.objective))
        for arc in self.arcs:
            if self.x[arc].varValue > 0.5:
                print(f"Arc ({arc[0] + (1 if one_indexed else 0)}, {arc[1] + (1 if one_indexed else 0)}) is in the"
                      f" solution with cost {round(self.cost_matrix[arc[0]][arc[1]],2)}")



## Spm. 2)
We can modify the TSP model such that the set that define the SEC's is empty. This means that we will allow for subtours in the solution.
This is done in the code snippet "STSP_no_SEC.S = []"

In [7]:
x_coords = "2.0 2.5 2.8 1.0 1.5 2.0 1.5 7.0 7.3 7.9 8.2 7.6".split(" ")
x_coords = [float(v) for v in x_coords]
y_coords = "3.0 3.5 2.7 8.0 8.5 8.0 7.5 5.0 5.4 5.6 5.0 4.5".split(" ")
y_coords = [float(v) for v in y_coords]
n = len(x_coords)

STSP_no_SEC = symmetric_tsp_dfj(n, x_coords=x_coords, y_coords=y_coords)
STSP_no_SEC.S = []
STSP_no_SEC.solve_and_print(quiet = False)

Status: Optimal
Objective value: 8.609660515461186
Arc (1, 2) is in the solution with cost 0.71
Arc (1, 3) is in the solution with cost 0.85
Arc (2, 3) is in the solution with cost 0.85
Arc (4, 5) is in the solution with cost 0.71
Arc (4, 7) is in the solution with cost 0.71
Arc (5, 6) is in the solution with cost 0.71
Arc (6, 7) is in the solution with cost 0.71
Arc (8, 9) is in the solution with cost 0.5
Arc (8, 12) is in the solution with cost 0.78
Arc (9, 10) is in the solution with cost 0.63
Arc (10, 11) is in the solution with cost 0.67
Arc (11, 12) is in the solution with cost 0.78


## Spm. 3)

We can retrive the SEC's by getting the model.constaints.items() and looking for the constraints with "Subtour" in the name. This is done in the code snippet "for name, constraint in STSPrelaxed.model.constraints.items(): if "Subtour" in name: print(name, " : ", constraint)"

In [8]:
x_coords = "2.0 2.5 2.8 1.0 1.5 2.0 1.5 7.0 7.3 7.9 8.2 7.6".split(" ")
x_coords = [float(v) for v in x_coords]
y_coords = "3.0 3.5 2.7 8.0 8.5 8.0 7.5 5.0 5.4 5.6 5.0 4.5".split(" ")
y_coords = [float(v) for v in y_coords]
n = len(x_coords)
S_1, S_2, S_3 = range(3), range(3, 7), range(7, 12)

STSPrelaxed = symmetric_tsp_dfj(n, x_coords=x_coords, y_coords=y_coords)
STSPrelaxed.S = [S_1, S_2, S_3]
STSPrelaxed.solve_and_print(quiet = False)

for name, constraint in STSPrelaxed.model.constraints.items():
    if "Subtour" in name:
        print(name, " : ", constraint)


Status: Optimal
Objective value: 21.347588159805703
Arc (1, 2) is in the solution with cost 0.71
Arc (1, 3) is in the solution with cost 0.85
Arc (2, 7) is in the solution with cost 4.12
Arc (3, 12) is in the solution with cost 5.13
Arc (4, 5) is in the solution with cost 0.71
Arc (4, 7) is in the solution with cost 0.71
Arc (5, 6) is in the solution with cost 0.71
Arc (6, 8) is in the solution with cost 5.83
Arc (8, 9) is in the solution with cost 0.5
Arc (9, 10) is in the solution with cost 0.63
Arc (10, 11) is in the solution with cost 0.67
Arc (11, 12) is in the solution with cost 0.78
SubtourElimination0  :  x_(0,_1) + x_(0,_2) + x_(1,_2) <= 2.0
SubtourElimination1  :  x_(3,_4) + x_(3,_5) + x_(3,_6) + x_(4,_5) + x_(4,_6) + x_(5,_6) <= 3.0
SubtourElimination2  :  x_(10,_11) + x_(7,_10) + x_(7,_11) + x_(7,_8) + x_(7,_9) + x_(8,_10) + x_(8,_11) + x_(8,_9) + x_(9,_10) + x_(9,_11) <= 4.0


## Spm. 4)
We add $S_2$ to the model and check which subtours are eliminated. We now get the subtour (4,5,6,4), so we see that all the subtours given are eliminated.

In [9]:
x_coords = "2.0 2.5 2.8 1.0 1.5 2.0 1.5 7.0 7.3 7.9 8.2 7.6".split(" ")
x_coords = [float(v) for v in x_coords]
y_coords = "3.0 3.5 2.7 8.0 8.5 8.0 7.5 5.0 5.4 5.6 5.0 4.5".split(" ")
y_coords = [float(v) for v in y_coords]
n = len(x_coords)
S_2 = range(3,7)

STSPrelaxed = symmetric_tsp_dfj(n, x_coords=x_coords, y_coords=y_coords)
STSPrelaxed.S = [S_2]
STSPrelaxed.solve_and_print(quiet = False)

Status: Optimal
Objective value: 16.139138366587915
Arc (1, 3) is in the solution with cost 0.85
Arc (1, 7) is in the solution with cost 4.53
Arc (2, 3) is in the solution with cost 0.85
Arc (2, 7) is in the solution with cost 4.12
Arc (4, 5) is in the solution with cost 0.71
Arc (4, 6) is in the solution with cost 1.0
Arc (5, 6) is in the solution with cost 0.71
Arc (8, 9) is in the solution with cost 0.5
Arc (8, 12) is in the solution with cost 0.78
Arc (9, 10) is in the solution with cost 0.63
Arc (10, 11) is in the solution with cost 0.67
Arc (11, 12) is in the solution with cost 0.78


## Spm. 5)
We can add all the SEC's from queation 2, and get a solution:

In [10]:
x_coords = "2.0 2.5 2.8 1.0 1.5 2.0 1.5 7.0 7.3 7.9 8.2 7.6".split(" ")
x_coords = [float(v) for v in x_coords]
y_coords = "3.0 3.5 2.7 8.0 8.5 8.0 7.5 5.0 5.4 5.6 5.0 4.5".split(" ")
y_coords = [float(v) for v in y_coords]
n = len(x_coords)
S_1, S_2, S_3 = range(3), range(3, 7), range(7, 12)

STSPrelaxed = symmetric_tsp_dfj(n, x_coords=x_coords, y_coords=y_coords)
STSPrelaxed.S = [S_1, S_2, S_3]
STSPrelaxed.solve_and_print(quiet = False)

Status: Optimal
Objective value: 21.347588159805703
Arc (1, 2) is in the solution with cost 0.71
Arc (1, 3) is in the solution with cost 0.85
Arc (2, 7) is in the solution with cost 4.12
Arc (3, 12) is in the solution with cost 5.13
Arc (4, 5) is in the solution with cost 0.71
Arc (4, 7) is in the solution with cost 0.71
Arc (5, 6) is in the solution with cost 0.71
Arc (6, 8) is in the solution with cost 5.83
Arc (8, 9) is in the solution with cost 0.5
Arc (9, 10) is in the solution with cost 0.63
Arc (10, 11) is in the solution with cost 0.67
Arc (11, 12) is in the solution with cost 0.78


We now get a solution with no subtours, thus it is the optimal solution to the STSP problem.